# Evaluation: paper-style aggregate throughput (Fig.7-like)

- **X**: experiment time (s)
- **Y**: **aggregate throughput** (Mbps), one line per algorithm (LIA / OLIA / ...)
- **Preferred source**: `[m]monitor bw=` (instantaneous), fallback to `path :* mean tp`
- **Background bands**: 3 phases from `tc_bw_*.log` (e.g. 0-50, 50-100, 100+)
- **In-band text**: link setting annotations (capacity / delay / loss)
- **Extra (last code cell)**: per-path time series + **steady-window means** from `parse_logs.phase_steady_windows_from_tc_bw` (10 s transition, CSV in `derived/`)
- **After the main Fig.7-style figure**: **byte-counted receive throughput** from pull lines `rcvim tp` (per-path 1 s window, summed) → `derived/evaluation_timeline_recv_bytetp.png` (not `GetBandwidthEstimate()`)

Set `RUN_DIR` / `METHOD_RUNS` in the config cell.

### One-click: Run All

Yes, you **can** run everything top-to-bottom.

- **Cursor / VS Code**: open this `.ipynb`, then toolbar **⋯ (more)** → **Run All**, or Command Palette (**Cmd+Shift+P**) → type **`Run All`** → choose **Run All Cells** / **Execute Notebook** (wording depends on version).
- **First time**: the first code cell may call `pip` (needs **network**); after packages exist it prints `OK` and costs almost nothing.
- **If Run All “stops” or spins forever**: the UI often highlights only the **first line** of a cell while the **whole** cell runs — check the cell just above (pip) or later imports. Use **Interrupt** if needed, fix path/network, **Restart kernel**, then Run All again.
- **Hard stop with `AssertionError`**: `RUN_DIR` must exist on this machine and contain `pull_*.log` and `tc_bw_*.log` (sync logs from the VM or change the path).

In [5]:
# Third-party deps for this notebook (needs network the first time you run it).
import importlib.util
import subprocess
import sys

_PACKAGES = ["pandas", "matplotlib", "numpy"]

def _have(mod: str) -> bool:
    return importlib.util.find_spec(mod) is not None

_missing = [p for p in _PACKAGES if not _have(p)]
if _missing:
    print("Installing (needs network):", ", ".join(_missing), flush=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-input",
            *_missing,
        ],
        timeout=600,
    )
    print("Installed:", ", ".join(_missing))
else:
    print("OK — already installed:", ", ".join(_PACKAGES))


OK — already installed: pandas, matplotlib, numpy


In [6]:
import os
import re
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

# Jupyter shows the spinner on line 1 even if the stall is later (pip / matplotlib / pandas).
# Force inline backend before pyplot so macOS doesn't sit on a GUI backend.
try:
    from IPython import get_ipython

    _ipython = get_ipython()
    if _ipython is not None:
        _ipython.run_line_magic("matplotlib", "inline")
except (ImportError, AttributeError):
    os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import pandas as pd


def find_repo() -> Path:
    """Cwd or parents until scripts/analyze/parse_logs.py exists."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p
    return start


REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts" / "analyze"))
import parse_logs as pl  # noqa: E402

print("REPO =", REPO)

REPO = /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment


In [7]:
# --- run directory (one vm_run folder with pull + tc_bw) ---
RUN_DIR = REPO / "logs_exp" / "log" / "vm_run_20260426_095810"  # <- change to run

# One method → one run dir. Keep in sync with RUN_DIR to avoid stale kernel state.
METHOD_RUNS = {
    # mpQUIC 4D-MAP T-model (not MPTCP ACCESS-T).
    "mpQUIC (T-model)": RUN_DIR,
}

PULL = next(RUN_DIR.glob("pull_*.log"), None)
TC_BW = next(RUN_DIR.glob("tc_bw_*.log"), None)
assert PULL and TC_BW, f"Need pull_*.log and tc_bw_*.log under {RUN_DIR}"
PULL, TC_BW

AssertionError: Need pull_*.log and tc_bw_*.log under /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/vm_run_20260426_095810

In [ ]:
from typing import Optional

_RE_MEAN_TP = re.compile(
    r"^(?P<date>\d{4}/\d{2}/\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) path :(?P<path>\d) mean tp: (?P<tp>[\d.]+)Mbps"
)


def load_total_tp_from_mean(pull_path: Path) -> pd.DataFrame:
    """Fallback: sum(path mean tp) per timestamp from pull log."""
    rows: list[dict] = []
    t0: Optional[float] = None
    with open(pull_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = _RE_MEAN_TP.match(line)
            if not m:
                continue
            try:
                tp = float(m["tp"])
            except ValueError:
                continue
            if not (tp == tp) or tp < 0:
                continue
            dt = datetime.strptime(m["date"] + " " + m["time"], "%Y/%m/%d %H:%M:%S")
            ts = dt.timestamp()
            if t0 is None:
                t0 = ts
            rows.append({"t_sec": ts - t0, "tp_mbps": tp})
    if not rows:
        return pd.DataFrame(columns=["t_sec", "tp_mbps_total", "source"])
    d = pd.DataFrame(rows)
    out = d.groupby("t_sec", as_index=False)["tp_mbps"].sum().rename(columns={"tp_mbps": "tp_mbps_total"})
    out["source"] = "mean_tp"
    return out


def load_total_tp_from_monitor(pull_path: Path, label: str = "") -> pd.DataFrame:
    """Preferred: sum monitor bw_bytes across paths, converted to Mbps."""
    _u, mon = pl.load_pull_log(pull_path, label=label)
    if mon.empty or "bw_bytes" not in mon.columns:
        return pd.DataFrame(columns=["t_sec", "tp_mbps_total", "source"])
    # Integer second t has many monitor samples per path — sum() would add them all and
    # inflate totals to 1e4+ "Mbps". Match per-path logic: mean per (t, path), then sum paths.
    d = mon[["t", "path", "bw_bytes"]].copy()
    d = d[d["bw_bytes"].notna()]
    if d.empty:
        return pd.DataFrame(columns=["t_sec", "tp_mbps_total", "source"])
    per_tp = d.groupby(["t", "path"], as_index=False)["bw_bytes"].mean()
    out = per_tp.groupby("t", as_index=False)["bw_bytes"].sum().rename(columns={"t": "t_sec"})
    out["tp_mbps_total"] = out["bw_bytes"] * 8.0 / 1e6
    out = out[["t_sec", "tp_mbps_total"]].sort_values("t_sec").reset_index(drop=True)
    out["source"] = "monitor_bw"
    return out


def load_method_series(run_dir: Path, label: str) -> tuple[pd.DataFrame, Path, Path]:
    pull = next(run_dir.glob("pull_*.log"), None)
    tc_bw = next(run_dir.glob("tc_bw_*.log"), None)
    assert pull is not None and tc_bw is not None, f"[{label}] missing pull/tc_bw in {run_dir}"

    ts = load_total_tp_from_monitor(pull, label=label)
    if ts.empty:
        ts = load_total_tp_from_mean(pull)
    assert not ts.empty, f"[{label}] no usable throughput rows in {pull}"
    ts = ts.copy()
    ts["label"] = label
    return ts, pull, tc_bw


def load_per_path_from_mean(pull_path: Path) -> pd.DataFrame:
    """Long-form: t_sec, path, tp_mbps (same t0 as aggregate mean-tp series)."""
    rows: list[dict] = []
    t0: Optional[float] = None
    with open(pull_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = _RE_MEAN_TP.match(line)
            if not m:
                continue
            try:
                tp = float(m["tp"])
            except ValueError:
                continue
            if not (tp == tp) or tp < 0:
                continue
            dt = datetime.strptime(m["date"] + " " + m["time"], "%Y/%m/%d %H:%M:%S")
            ts = dt.timestamp()
            if t0 is None:
                t0 = ts
            rows.append(
                {"t_sec": ts - t0, "path": int(m["path"]), "tp_mbps": tp}
            )
    if not rows:
        return pd.DataFrame(columns=["t_sec", "path", "tp_mbps"])
    d = pd.DataFrame(rows)
    return d.groupby(["t_sec", "path"], as_index=False)["tp_mbps"].mean()


def load_per_path_from_monitor(pull_path: Path, label: str = "") -> pd.DataFrame:
    """Long-form: t_sec, path, tp_mbps from [m]monitor bw."""
    _u, mon = pl.load_pull_log(pull_path, label=label)
    if mon.empty or "bw_bytes" not in mon.columns:
        return pd.DataFrame(columns=["t_sec", "path", "tp_mbps"])
    d = mon[["t", "path", "bw_bytes"]].dropna(subset=["bw_bytes"])
    d = d.rename(columns={"t": "t_sec"})
    d["tp_mbps"] = d["bw_bytes"] * 8.0 / 1e6
    return d.groupby(["t_sec", "path"], as_index=False)["tp_mbps"].mean()


def load_per_path_timeseries(pull_path: Path, label: str) -> pd.DataFrame:
    """Match aggregate source: monitor first, else mean-tp per path."""
    p = load_per_path_from_monitor(pull_path, label=label)
    if not p.empty:
        return p
    return load_per_path_from_mean(pull_path)

In [ ]:
# --- paper-style controls ---
SMOOTH = 3             # rolling window (samples)
X_MIN = 10.0           # paper-style often starts around 10s
Y_LIM = None           # auto-scale by default; set (25.0, 50.0) for paper-like fixed range

if "METHOD_RUNS" not in globals() or not METHOD_RUNS:
    # Fallback so this cell won't crash if config cell wasn't run yet.
    METHOD_RUNS = {
        "mpQUIC (T-model)": REPO / "logs_exp" / "log" / "vm_run_20260426_095810",
    }
    print("[warn] METHOD_RUNS was undefined. Using fallback. Please edit METHOD_RUNS cell for full multi-method plot.")

method_rows = []
method_logs = {}
for label, run_dir in METHOD_RUNS.items():
    ts, pull, tc_bw = load_method_series(run_dir, label)
    if SMOOTH and SMOOTH > 1:
        ts = ts.copy()
        ts["tp_mbps_total"] = ts["tp_mbps_total"].rolling(SMOOTH, min_periods=1).mean()
    method_rows.append(ts)
    method_logs[label] = {"pull": pull, "tc_bw": tc_bw, "source": ts["source"].iloc[0]}

all_ts = pd.concat(method_rows, ignore_index=True)
t_max = float(all_ts["t_sec"].max()) if not all_ts.empty else 0.0
all_ts.head()

In [ ]:
# --- tc steps for background phases (use first method as reference) ---
# Also align all timelines to tc start, so boundaries are at 0/50/100s.
ref_label = list(METHOD_RUNS.keys())[0]
ref_pull = method_logs[ref_label]["pull"]
ref_tc = method_logs[ref_label]["tc_bw"]

tc_df = pl.tc_bw_with_pull_t(ref_tc, ref_pull).sort_values("at_sec").reset_index(drop=True)

if not tc_df.empty and tc_df["t_pull"].notna().any():
    at0 = tc_df[tc_df["at_sec"] == 0]
    tc_t0_pull = float(at0.iloc[0]["t_pull"]) if not at0.empty else float(tc_df["t_pull"].min())
else:
    tc_t0_pull = 0.0

all_ts = all_ts.copy()
all_ts["t_plot"] = all_ts["t_sec"] - tc_t0_pull

tc_df = tc_df.copy()
tc_df["t_plot"] = tc_df["t_pull"] - tc_t0_pull

t_max = float(all_ts["t_plot"].max()) if not all_ts.empty else 0.0
print(f"tc_start_shift = {tc_t0_pull:.3f}s (pull-time -> tc-time)")
tc_df[["at_sec", "t_pull", "t_plot", "bw_mbit", "dev"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))
band_colors = ["#e8eacd", "#ecd9ee", "#dff0df", "#d9e7f7"]

# Optional text in each phase (edit to your exact scenario text)
PHASE_TEXTS = [
    "Link1:20Mbps,40ms,0%\nLink2:20Mbps,20ms,0.001%",
    "Link1:20Mbps,40ms,0%\nLink2:30Mbps,20ms,0.001%",
    "Link1:20Mbps,40ms,0%\nLink2:10Mbps,20ms,0.001%",
]

# Phase shading + step lines (already shifted to tc-start timebase)
segments = []
if not tc_df.empty and tc_df["t_plot"].notna().any():
    n = len(tc_df)
    for i in range(n):
        a = float(tc_df.loc[i, "t_plot"])
        b = float(tc_df.loc[i + 1, "t_plot"]) if i + 1 < n else t_max + 1.0
        b = min(b, t_max + 0.1)
        if b <= a:
            continue
        bw = float(tc_df.loc[i, "bw_mbit"])
        dev = str(tc_df.loc[i, "dev"])
        segments.append((a, b, bw, dev))
        ax.axvspan(a, b, color=band_colors[i % len(band_colors)], alpha=0.45, zorder=0)
        ax.axvline(a, color="#555", ls="--", lw=0.8, alpha=0.6, zorder=1)

from matplotlib.transforms import blended_transform_factory

_txt_tr = blended_transform_factory(ax.transData, ax.transAxes)
for i, (a, b, bw, dev) in enumerate(segments):
    txt = PHASE_TEXTS[i] if i < len(PHASE_TEXTS) else f"{dev}: {bw:.0f}Mbps"
    xm = 0.5 * (a + b)
    ax.text(
        xm,
        0.04,
        txt,
        transform=_txt_tr,
        ha="center",
        va="bottom",
        fontsize=8,
        color="#222",
        linespacing=1.1,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            alpha=0.9,
            edgecolor="#bbb",
            linewidth=0.6,
        ),
        zorder=5,
    )

# Multi-method aggregate throughput lines
markers = ["o", "^", "s", "D", "v", "x", "*"]
for i, label in enumerate(METHOD_RUNS.keys()):
    d = all_ts[all_ts["label"] == label].sort_values("t_plot")
    ax.plot(
        d["t_plot"],
        d["tp_mbps_total"],
        label=label,
        lw=1.1,
        marker=markers[i % len(markers)],
        ms=4,
        markevery=max(len(d) // 16, 1),
        alpha=0.95,
        zorder=3,
    )

ax.set_xlabel(
    "Time since tc start (s)",
    fontsize=14,
    labelpad=6,
)
ax.set_ylabel("Throughput (Mbps)", fontsize=14)
ax.set_title("Performance when link capacity changes", fontsize=15)
ax.set_xlim(X_MIN, max(t_max * 1.02, X_MIN + 1))
if Y_LIM is not None:
    ax.set_ylim(*Y_LIM)
    y_min = float(all_ts["tp_mbps_total"].min())
    y_max = float(all_ts["tp_mbps_total"].max())
    if y_max < Y_LIM[0] or y_min > Y_LIM[1]:
        print(f"[warn] Data range [{y_min:.3f}, {y_max:.3f}] is outside Y_LIM={Y_LIM}. Set Y_LIM=None to see lines.")
ax.grid(True, alpha=0.15, zorder=2)
ax.legend(loc="upper center", ncol=4, fontsize=11, framealpha=0.95)

fig.tight_layout()
fig.subplots_adjust(bottom=0.18)
out_png = REPO / "derived" / "evaluation_timeline_notebook.png"
out_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_png, dpi=160)
plt.show()
print("Saved:", out_png)
print("Data source per method:", {k: v["source"] for k, v in method_logs.items()})

In [ ]:
# Byte-counted receive rate (quic `GetStatisticstp` recv) — re-plot

# Parses **recv-only** `path :… mean tp: …, rcvim tp: …` lines. `rcvim` is the per-1s window receive rate
# from `thistimerecvsize` in `quic-go43DMAP/ackhandler/received_packet_handler.go` (not CC `GetBandwidthEstimate`).
# Aggregate = sum of `rcvim` over paths (same time base as main figure: `t_plot` = pull `t` minus `tc_t0_pull`).

# Requires the main figure cell to have run (defines `segments`, `PHASE_TEXTS`, `band_colors`).

import re
from pathlib import Path

from IPython.display import display
from matplotlib.transforms import blended_transform_factory

_RE_RCVIM_RECV = re.compile(
    r"^(?P<date>\d{4}/\d{2}/\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) path :(?P<path>\d+) "
    r"mean tp: (?P<mean>NaN|nan|[\d.]+)Mbps, rcvim tp: (?P<rcvim>[\d.]+)Mbps"
)


def load_total_recv_rcvim_mbps(pull_path: Path) -> pd.DataFrame:
    """Per-second aggregate receive throughput (sum over paths) from rcvim tp lines."""
    t0 = int(pl._first_metric_tod_sec(pull_path))
    rows: list[dict] = []
    hms = pl._hms_to_sec
    with open(pull_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = _RE_RCVIM_RECV.search(line)
            if not m:
                continue
            try:
                r = float(m["rcvim"])
            except ValueError:
                continue
            if r < 0 or r != r:
                continue
            ts = int(hms(m["date"], m["time"])) - t0
            rows.append({"t_sec": float(ts), "path": int(m["path"]), "rcvim_mbps": r})
    if not rows:
        return pd.DataFrame(columns=["t_sec", "tp_mbps_total", "source", "label"])
    d = pd.DataFrame(rows)
    per = d.groupby(["t_sec", "path"], as_index=False)["rcvim_mbps"].mean()
    tot = per.groupby("t_sec", as_index=False)["rcvim_mbps"].sum().rename(
        columns={"rcvim_mbps": "tp_mbps_total"}
    )
    tot["source"] = "recv_rcvim"
    return tot


recv_rows: list[pd.DataFrame] = []
for label, run_dir in METHOD_RUNS.items():
    pull = next(run_dir.glob("pull_*.log"), None)
    assert pull, f"[{label}] no pull_*.log"
    r = load_total_recv_rcvim_mbps(pull)
    if r.empty:
        print(f"[warn] [{label}] no 'rcvim tp' lines in {pull} — is QUIC info logging on?")
        continue
    r = r.copy()
    r["label"] = label
    if SMOOTH and SMOOTH > 1:
        r["tp_mbps_total"] = r["tp_mbps_total"].rolling(SMOOTH, min_periods=1).mean()
    recv_rows.append(r)

if recv_rows:
    all_ts_recv = pd.concat(recv_rows, ignore_index=True)
    all_ts_recv["t_plot"] = all_ts_recv["t_sec"] - float(tc_t0_pull)
    t_max_r = float(all_ts_recv["t_plot"].max())
    figr, axr = plt.subplots(figsize=(11, 5.2))
    for i, (a, b, bw, dev) in enumerate(segments):
        c = band_colors[i % len(band_colors)]
        axr.axvspan(a, b, color=c, alpha=0.45, zorder=0)
        axr.axvline(a, color="#555", ls="--", lw=0.8, alpha=0.6, zorder=1)
    _txt_r = blended_transform_factory(axr.transData, axr.transAxes)
    for i, (a, b, bw, dev) in enumerate(segments):
        txt = PHASE_TEXTS[i] if i < len(PHASE_TEXTS) else f"{dev}: {bw:.0f}Mbps"
        xm = 0.5 * (a + b)
        axr.text(
            xm, 0.04, txt, transform=_txt_r, ha="center", va="bottom", fontsize=8, color="#222",
            linespacing=1.1,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9, edgecolor="#bbb", linewidth=0.6),
            zorder=5,
        )
    markers = ["o", "^", "s", "D", "v", "x", "*"]
    for i, label in enumerate(METHOD_RUNS.keys()):
        d = all_ts_recv[all_ts_recv["label"] == label].sort_values("t_plot")
        if d.empty:
            continue
        axr.plot(
            d["t_plot"],
            d["tp_mbps_total"],
            label=label,
            lw=1.1,
            marker=markers[i % len(markers)],
            ms=4,
            markevery=max(len(d) // 16, 1),
            alpha=0.95,
            zorder=3,
        )
    axr.set_xlabel("Time since tc start (s)", fontsize=14, labelpad=6)
    axr.set_ylabel("Receive throughput (Mbps)", fontsize=14)
    axr.set_title("Performance when link capacity changes (byte-counted recv, sum paths)", fontsize=15)
    axr.set_xlim(X_MIN, max(t_max_r * 1.02, X_MIN + 1))
    if Y_LIM is not None:
        axr.set_ylim(*Y_LIM)
    axr.grid(True, alpha=0.15, zorder=2)
    axr.legend(loc="upper center", ncol=4, fontsize=11, framealpha=0.95)
    figr.tight_layout()
    figr.subplots_adjust(bottom=0.18)
    out_r = REPO / "derived" / "evaluation_timeline_recv_bytetp.png"
    out_r.parent.mkdir(parents=True, exist_ok=True)
    figr.savefig(out_r, dpi=160)
    display(figr)
    print("Saved:", out_r)
    print("Y-axis: sum of per-path `rcvim tp` (1 s window) from pull log; not CC BandwidthEstimate.")
else:
    print("Skip: no rcvim data — enable pull logging with `path :* rcvim` lines (QUIC info).")


### Congestion / learn (same time base as the main figure)

- **X-axis** `t_plot` = *time since first `[utility]/[m]monitor` in pull* minus `tc_t0_pull` → **aligns with** “Time since tc start (s)”.
- **Gain / backoff** from `[utility]` (OLIA is driven via `SetUtilityControl`). Default path: **1** (learn leader) if present, else **0**; one point per `t` via `groupby(...).last()`.
- **wT, wD, wL** from `[learn]` (only in `mode=learn`). If no `[learn]` lines, the second figure is skipped.
- **Outputs**: figures are **shown under this cell** (`display(fig)`), and saved under `derived/` as `evaluation_cc_gain_backoff.png` and `evaluation_learn_weights.png`.

In [ ]:
# --- Gain / backoff and learned weights vs t_plot (timestamp base = same as main fig) ---

from IPython.display import display

ref_label = list(METHOD_RUNS.keys())[0]
ref_pull = method_logs[ref_label]["pull"]
df_g = pl.load_utility_gains(ref_pull, label=ref_label)
df_learn = pl.load_learn_from_pull(ref_pull, label=ref_label)

# Prefer path 1 (learn leader in quic stack); else 0
PREF = 1 if not df_g.empty and (df_g["path"] == 1).any() else 0
sub = df_g[df_g["path"] == PREF] if not df_g.empty else df_g
if not sub.empty:
    sub = sub.copy()
    sub["t_plot"] = sub["t"].astype(float) - float(tc_t0_pull)
    gb = sub.groupby("t", as_index=False).agg({"gain": "last", "backoff": "last"})
    gb["t_plot"] = gb["t"].astype(float) - float(tc_t0_pull)
    gb = gb[pd.to_numeric(gb["gain"], errors="coerce").notna() & pd.to_numeric(gb["backoff"], errors="coerce").notna()]

    fig1, ax1 = plt.subplots(figsize=(11, 3.8))
    ax1.plot(
        gb["t_plot"],
        gb["gain"],
        lw=0.9,
        label="gain",
        color="C0",
    )
    ax1.plot(
        gb["t_plot"],
        gb["backoff"],
        lw=0.9,
        label="backoff",
        color="C1",
    )
    if not tc_df.empty and tc_df["t_plot"].notna().any():
        n = len(tc_df)
        for i in range(n):
            a = float(tc_df.loc[i, "t_plot"])
            ax1.axvline(a, color="#555", ls="--", lw=0.8, alpha=0.5)
    ax1.set_xlabel("Time since tc start (s)")
    ax1.set_ylabel("gain / backoff (utility → OLIA)")
    ax1.set_title(f"Utility gain & backoff (path={PREF}, {ref_label})")
    ax1.grid(True, alpha=0.2)
    ax1.legend(loc="upper right", ncol=2)
    fig1.tight_layout()
    p1 = REPO / "derived" / "evaluation_cc_gain_backoff.png"
    fig1.savefig(p1, dpi=160)
    display(fig1)
    plt.close(fig1)
    print("Saved:", p1)
else:
    print("No [utility] gain/backoff; skip figure.")

if not df_learn.empty:
    sL = df_learn[df_learn["path"] == 1] if (df_learn["path"] == 1).any() else df_learn
    sL = sL.copy()
    sL["t_plot"] = sL["t"].astype(float) - float(tc_t0_pull)
    wplot = sL.groupby("t", as_index=False).agg(
        wT=("wT", "last"), wD=("wD", "last"), wL=("wL", "last")
    )
    wplot["t_plot"] = wplot["t"].astype(float) - float(tc_t0_pull)

    fig2, ax2 = plt.subplots(figsize=(11, 3.8))
    ax2.plot(wplot["t_plot"], wplot["wT"], label="wT", lw=1.0)
    ax2.plot(wplot["t_plot"], wplot["wD"], label="wD", lw=1.0)
    ax2.plot(wplot["t_plot"], wplot["wL"], label="wL", lw=1.0)
    if not tc_df.empty and tc_df["t_plot"].notna().any():
        n = len(tc_df)
        for i in range(n):
            a = float(tc_df.loc[i, "t_plot"])
            ax2.axvline(a, color="#555", ls="--", lw=0.8, alpha=0.5)
    ax2.set_xlabel("Time since tc start (s)")
    ax2.set_ylabel("weight")
    ax2.set_title(f"Learned (wT,wD,wL) on simplex — {ref_label}")
    ax2.set_ylim(0, 1.02)
    ax2.grid(True, alpha=0.2)
    ax2.legend(loc="upper right", ncol=3)
    fig2.tight_layout()
    p2 = REPO / "derived" / "evaluation_learn_weights.png"
    fig2.savefig(p2, dpi=160)
    display(fig2)
    plt.close(fig2)
    print("Saved:", p2)
else:
    print("No [learn] lines (not learn mode, or log missing); skip weights figure.")

In [ ]:
# --- Per-path throughput + phase steady means (ref run = first METHOD_RUNS entry) ---
# Requires cells above: method_logs, ref_label, tc_t0_pull, SMOOTH, all_ts, tc_df, t_max (tc-time)

import numpy as np

ref_label = list(METHOD_RUNS.keys())[0]
ref_pull = method_logs[ref_label]["pull"]
ref_tc = method_logs[ref_label]["tc_bw"]
_src = method_logs[ref_label]["source"]

pp_long = load_per_path_timeseries(ref_pull, ref_label)
if pp_long.empty:
    print("No per-path series; check pull log for [m]monitor or mean tp lines.")
else:
    if SMOOTH and SMOOTH > 1:
        pp_long = pp_long.sort_values(["path", "t_sec"])
        pp_long = pp_long.copy()
        pp_long["tp_mbps"] = (
            pp_long.groupby("path")["tp_mbps"]
            .transform(lambda s: s.rolling(SMOOTH, min_periods=1).mean())
        )
    pp_long["t_plot"] = pp_long["t_sec"] - tc_t0_pull

    # ---- Fig A: per-path lines (tc-time) ----
    band_colors = ["#e8eacd", "#ecd9ee", "#dff0df", "#d9e7f7"]
    fig_a, ax_a = plt.subplots(figsize=(11, 4.5))
    if not tc_df.empty and tc_df["t_plot"].notna().any():
        n = len(tc_df)
        for i in range(n):
            a = float(tc_df.loc[i, "t_plot"])
            b = float(tc_df.loc[i + 1, "t_plot"]) if i + 1 < n else t_max + 1.0
            b = min(b, t_max + 0.1)
            if b > a:
                ax_a.axvspan(
                    a, b,
                    color=band_colors[i % len(band_colors)],
                    alpha=0.45,
                    zorder=0,
                )
    for p_id in sorted(pp_long["path"].unique()):
        d = pp_long[pp_long["path"] == p_id].sort_values("t_plot")
        ax_a.plot(
            d["t_plot"],
            d["tp_mbps"],
            label=f"path {p_id}",
            lw=1.0,
            zorder=2,
        )
    # No overlay of aggregate "total" here: if that series is on a very different
    # scale (e.g. wrong unit in an old run), Matplotlib y-autoscale 0-1e4+ and
    # per-path curves (~2-25 Mbps) look stuck at 0. Total stays in the top cell.
    ax_a.set_xlabel("Time since tc start (s)")
    ax_a.set_ylabel("Throughput (Mbps)")
    ax_a.set_title(
        f"Per-path throughput (source={_src}, ref={ref_label})"
    )
    ax_a.set_xlim(X_MIN, max(t_max * 1.02, X_MIN + 1))
    yhi = max(0.5, float(pp_long["tp_mbps"].max()) * 1.12)
    ax_a.set_ylim(0, yhi)
    ax_a.grid(True, alpha=0.15)
    ax_a.legend(loc="upper right", fontsize=9)
    fig_a.tight_layout()
    out_a = REPO / "derived" / "evaluation_per_path_notebook.png"
    out_a.parent.mkdir(parents=True, exist_ok=True)
    fig_a.savefig(out_a, dpi=160)
    plt.show()
    print("Saved:", out_a)

    # ---- Phase steady windows (pull-time) + bar chart of means ----
    _tend = max(220.0, float(all_ts["t_sec"].max()) + 5.0)
    ph = pl.phase_steady_windows_from_tc_bw(
        ref_tc,
        ref_pull,
        transition_sec=10.0,
        experiment_end_sec=_tend,
    )
    print("Steady windows (skip 10s after each tc step):")
    try:
        from IPython.display import display

        display(ph)
    except Exception:
        print(ph)

    rows_mean = []
    for _, r in ph.iterrows():
        pnum = int(r["phase"])
        lo, hi = float(r["t_steady_start"]), float(r["t_steady_end"])
        cap = float(r["bw_mbit"])
        w = (pp_long["t_sec"] >= lo) & (pp_long["t_sec"] < hi)
        sub = pp_long.loc[w]
        for path_id, g in sub.groupby("path"):
            rows_mean.append(
                {
                    "phase": pnum,
                    "cap_mbit": cap,
                    "path": int(path_id),
                    "tp_mean_mbps": float(g["tp_mbps"].mean()) if not g.empty else float("nan"),
                }
            )
        wt = (all_ts["t_sec"] >= lo) & (all_ts["t_sec"] < hi) & (all_ts["label"] == ref_label)
        tsub = all_ts.loc[wt]
        rows_mean.append(
            {
                "phase": pnum,
                "cap_mbit": cap,
                "path": -1,
                "tp_mean_mbps": float(tsub["tp_mbps_total"].mean()) if not tsub.empty else float("nan"),
            }
        )
    ph_mean = pd.DataFrame(rows_mean)
    if not ph_mean.empty:
        csv_p = REPO / "derived" / "evaluation_phase_steady_means.csv"
        ph_mean.to_csv(csv_p, index=False)
        print("Wrote", csv_p)

    # Bar chart: grouped by phase, one bar per path + total (aligned for missing phases)
    if not ph_mean.empty:
        paths_ord = sorted([p for p in ph_mean["path"].unique() if p >= 0])
        if -1 in ph_mean["path"].values:
            paths_ord = paths_ord + [-1]
        phases_list = sorted(ph_mean["phase"].unique().astype(int).tolist())
        n_ph = len(phases_list)
        x = np.arange(n_ph, dtype=float)
        n_bars = len(paths_ord)
        w = 0.8 / max(n_bars, 1)
        fig_b, ax_b = plt.subplots(figsize=(9, 4.2))
        for i, p_id in enumerate(paths_ord):
            lab = "total" if p_id == -1 else f"path {p_id}"
            vals = []
            for ph_i in phases_list:
                row = ph_mean[(ph_mean["phase"] == ph_i) & (ph_mean["path"] == p_id)]
                if row.empty or row["tp_mean_mbps"].isna().all():
                    vals.append(np.nan)
                else:
                    vals.append(float(row["tp_mean_mbps"].iloc[0]))
            offset = (i - (n_bars - 1) / 2.0) * w
            ax_b.bar(x + offset, vals, width=w * 0.9, label=lab)
        ax_b.set_xlabel("Phase (steady window after 10s transition)")
        ax_b.set_ylabel("Mean throughput (Mbps)")
        ax_b.set_title(
            "Steady-state mean (per path + total) — from phase_steady_windows_from_tc_bw"
        )
        tick_labels = []
        for ph_i in phases_list:
            c = ph_mean[ph_mean["phase"] == ph_i]["cap_mbit"].drop_duplicates().iloc[0]
            tick_labels.append(f"P{ph_i}\n({c:.0f} M cap)")
        ax_b.set_xticks(x)
        ax_b.set_xticklabels(tick_labels, fontsize=9)
        ax_b.grid(True, axis="y", alpha=0.2)
        ax_b.legend(fontsize=8, ncol=2, loc="upper right")
        fig_b.tight_layout()
        out_b = REPO / "derived" / "evaluation_phase_steady_bars.png"
        fig_b.savefig(out_b, dpi=160)
        plt.show()
        print("Saved:", out_b)


## Notes

- This notebook now targets **paper Fig.7 style**: aggregate throughput + multi-method overlays.
- Throughput source is `monitor bw_bytes` first (more instantaneous); if absent, it falls back to `mean tp` parsing.
- **CC + learn (after main fig, same `t_plot`)** writes `derived/evaluation_cc_gain_backoff.png` (utility **gain/backoff** → OLIA) and, if `mode=learn`, `derived/evaluation_learn_weights.png` (**wT,wD,wL** from `[learn]`).
- **Per-path + steady means** (next section) writes `derived/evaluation_per_path_notebook.png`, `evaluation_phase_steady_bars.png`, and `evaluation_phase_steady_means.csv`. The per-path plot **no longer overlays** aggregate total: mixed scales or wrong-unit totals can squash the y-axis. Total is only in the first figure.
- Keep phase text (`PHASE_TEXTS`) aligned with your actual experiment profile.
- To reproduce the full paper legend, fill all methods in `METHOD_RUNS` with their own `vm_run_*` folders.